# Hierarchical (Ward) Clustering — Non-Spatial

Agglomerative clustering with **Ward linkage** applied to the same two feature sets used in `skater_clustering.ipynb` and `kmeans_clustering.ipynb`, without any spatial constraint.  Ward linkage minimises within-cluster variance at each merge step — the same objective as SKATER — so the comparison directly shows what geographic contiguity costs in feature-space quality.

| # | Features | Interpretation |
|---|----------|----------------|
| 1 | `lp3_scale`, `lp3_skew` | Shape of the flood frequency curve |
| 2 | `log10(1/aep)` for action / flood / moderate / major | How rare local flood impacts are |

Workflow per analysis:
1. Build the full Ward linkage matrix (`scipy.cluster.hierarchy.linkage`) once — used for the dendrogram and silhouette sweep.
2. Sweep k=2–14 via `fcluster` (no re-fitting needed).
3. Final labels from `AgglomerativeClustering` at the chosen k.
4. Compare against saved SKATER labels with ARI, NMI, and a spatial agreement map.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rplot

plt.style.use('ryan')

from pathlib import Path
from shapely.geometry import Point
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.optimize import linear_sum_assignment

## Configuration

In [ ]:
DATA_DIR   = Path("/home/ryan/data/flood_hazard")
GAGES2_DIR = Path("/home/ryan/data/usgs/GAGES_2/basinchar_and_report_sept_2011/spreadsheets-in-csv-format")
SKATER_DIR = Path(".")

N_CLUSTERS_LP3    = 8
N_CLUSTERS_AEP    = 8
MAX_DISTURB_INDEX = 15
WINSOR            = 0.02
LINKAGE_METHOD    = 'ward'
SAVEFIG           = False

CONUS_EXTENT = [-125, -66, 24, 50]

def make_conus_ax(fig, pos=111, title=''):
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,     facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,    linewidth=0.4, zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.6, zorder=3)
    if title:
        ax.set_title(title)
    return ax

def read_skater(path):
    df = pd.read_csv(path)[["site_no", "cluster"]]
    df["site_no"] = df["site_no"].astype(str).str.zfill(8)
    return df

## Load shared data

In [ ]:
ffa  = pd.read_parquet(DATA_DIR / "ffa" / "flood_frequency.parquet")
meta = pd.read_parquet(DATA_DIR / "metadata" / "site_info.parquet")[["site_no", "latitude", "longitude"]]

gages2 = pd.read_csv(GAGES2_DIR / "conterm_bas_classif.txt", encoding="latin1")
gages2["site_no"] = gages2["STAID"].astype(str).str.zfill(8)

print(f"FFA records : {len(ffa):,}")
print(f"Site meta   : {len(meta):,}")
print(f"GAGES-2     : {len(gages2):,}")

---
# Analysis 1: LP3 Hierarchical Clustering

Features: **`lp3_scale`** and **`lp3_skew`** — same filtering, winsorization, and RobustScaler as the SKATER and K-Means notebooks.

In [ ]:
df1 = (
    ffa[ffa.record_ok & ffa.lp3_scale.notna() & ffa.lp3_skew.notna()]
    [["site_no", "lp3_scale", "lp3_skew"]]
    .merge(meta, on="site_no")
    .merge(gages2[["site_no", "HYDRO_DISTURB_INDX"]], on="site_no", how="left")
)
df1 = df1[
    df1.HYDRO_DISTURB_INDX.notna() & (df1.HYDRO_DISTURB_INDX <= MAX_DISTURB_INDEX)
].reset_index(drop=True)

print(f"LP3 analysis: {len(df1):,} sites")

## Winsorize and scale features

In [ ]:
LP3_FEATS  = ["lp3_scale", "lp3_skew"]
LP3_SCALED = [f + "_s" for f in LP3_FEATS]

df1_clip = df1.copy()
for col in LP3_FEATS:
    lo, hi = df1[col].quantile([WINSOR, 1 - WINSOR])
    df1_clip[col] = df1[col].clip(lo, hi)
    print(f"{col}: clipped [{lo:.4f}, {hi:.4f}]")

scaler1 = RobustScaler()
X1 = scaler1.fit_transform(df1_clip[LP3_FEATS])
for i, col in enumerate(LP3_SCALED):
    df1[col] = X1[:, i]

## Build Ward linkage matrix and dendrogram — LP3

A truncated dendrogram (last 50 merges) gives a visual sense of where the natural cut levels are.

In [ ]:
Z1 = linkage(X1, method=LINKAGE_METHOD)

fig, ax = plt.subplots(figsize=(13, 5))
dendrogram(
    Z1, ax=ax,
    truncate_mode='lastp', p=50,
    leaf_rotation=90, leaf_font_size=8,
    color_threshold=0,
    above_threshold_color=rplot.OKABE_ITO[0]
)
ax.set_title(f'LP3 Ward dendrogram (last 50 merges)  —  {len(df1):,} sites')
ax.set_xlabel('Site (or cluster size)')
ax.set_ylabel('Ward linkage distance')
plt.tight_layout()
plt.show()

## Silhouette sweep — LP3

Cut the pre-computed linkage matrix at k=2–14 via `fcluster` (no re-fitting).

In [ ]:
sil_k_range = range(2, 15)
sil_scores1 = []

for k in sil_k_range:
    labels = fcluster(Z1, k, criterion='maxclust') - 1
    sil_scores1.append(silhouette_score(X1, labels))
    print(f"k={k:2d}  sil={sil_scores1[-1]:.4f}")

best_k1 = list(sil_k_range)[np.argmax(sil_scores1)]
print(f"\nBest k by silhouette: {best_k1}  (score={max(sil_scores1):.4f})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(sil_k_range), sil_scores1, marker='o', color=rplot.OKABE_ITO[0])
ax.axvline(best_k1, color='red', linestyle='--', label=f'best k={best_k1}')
ax.axvline(N_CLUSTERS_LP3, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_LP3}')
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Mean silhouette score')
ax.set_title('Silhouette — LP3 Ward')
ax.legend()
plt.tight_layout()
plt.show()

## Elbow plot — LP3

Total within-cluster variance (sum of Ward merge distances for all merges up to k) vs k.  Each point on the linkage matrix encodes the incremental WCSS increase at that merge, so `inertia[k] = sum(Z[:n−k, 2])`.

In [ ]:
n1 = len(X1)
inertia1 = [Z1[:n1 - k, 2].sum() for k in sil_k_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(sil_k_range), inertia1, marker='o', color=rplot.OKABE_ITO[0])
ax.axvline(best_k1, color='red',  linestyle='--', label=f'best sil k={best_k1}')
ax.axvline(N_CLUSTERS_LP3, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_LP3}')
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Total within-cluster variance (inertia)')
ax.set_title('Elbow — LP3 Ward')
ax.legend()
plt.tight_layout()
plt.show()

## Hierarchical clustering — LP3

Using `N_CLUSTERS_LP3` to match SKATER for a direct comparison.

In [ ]:
hc1 = AgglomerativeClustering(n_clusters=N_CLUSTERS_LP3, linkage=LINKAGE_METHOD)
df1["cluster"] = hc1.fit_predict(X1) + 1

sizes1 = df1["cluster"].value_counts().sort_index()
print(f"N_CLUSTERS={N_CLUSTERS_LP3}  |  linkage={LINKAGE_METHOD}")
print(f"Silhouette: {silhouette_score(X1, df1['cluster']):+.4f}")
print(sizes1.to_string())
print(f"\nMin: {sizes1.min()}  |  Max: {sizes1.max()}  |  Ratio: {sizes1.max()/sizes1.min():.1f}x")

## Cluster map — LP3

In [ ]:
cluster_colors1 = rplot.cluster_cmap(N_CLUSTERS_LP3).colors

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f'LP3 Ward hierarchical clusters  (N={N_CLUSTERS_LP3})\n'
    f'Features: lp3_scale, lp3_skew'
))

for cl in sorted(df1["cluster"].unique()):
    idx = df1["cluster"] == cl
    ax.scatter(
        df1.loc[idx, "longitude"], df1.loc[idx, "latitude"],
        s=20, color=cluster_colors1[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(),
        label=f'C{cl} (n={idx.sum():,})'
    )

ax.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)
plt.tight_layout()
plt.show()

## Cluster profiles — LP3

In [ ]:
profile1 = (
    df1.groupby("cluster")[["lp3_scale", "lp3_skew"]]
    .agg(["mean", "std", "count"])
)
profile1.columns = ["_".join(c) for c in profile1.columns]
print(profile1.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, feat in enumerate(LP3_FEATS):
    means = profile1[f"{feat}_mean"]
    stds  = profile1[f"{feat}_std"]
    axes[i].bar(
        means.index, means.values,
        yerr=stds.values, capsize=4,
        color=[cluster_colors1[c - 1] for c in means.index],
        edgecolor='none'
    )
    axes[i].set_xlabel('Cluster')
    axes[i].set_title(feat)
rplot.panel_labels(list(axes))
plt.suptitle('LP3 Ward: cluster mean ± SD', y=1.01)
plt.tight_layout()
plt.show()

---
# Analysis 2: AEP Threshold Hierarchical Clustering

Features: **log₁₀ return period** for each of the four NWS flood threshold levels — action, flood, moderate, major.

In [ ]:
AEP_COLS = ["action_aep", "flood_aep", "moderate_aep", "major_aep"]

df2 = (
    ffa[ffa.record_ok & ~ffa.degenerate_fit]
    [["site_no"] + AEP_COLS]
    .dropna(subset=AEP_COLS)
    .merge(meta, on="site_no")
    .merge(gages2[["site_no", "HYDRO_DISTURB_INDX"]], on="site_no", how="left")
)
df2 = df2[
    df2.HYDRO_DISTURB_INDX.notna() & (df2.HYDRO_DISTURB_INDX <= MAX_DISTURB_INDEX)
].reset_index(drop=True)

print(f"AEP analysis: {len(df2):,} sites")

## Transform and scale features

In [ ]:
RP_FEATS  = ["action_rp", "flood_rp", "moderate_rp", "major_rp"]
RP_SCALED = [f + "_s" for f in RP_FEATS]

for aep_col, rp_col in zip(AEP_COLS, RP_FEATS):
    df2[rp_col] = np.log10(1.0 / df2[aep_col].clip(lower=1e-6))

df2_clip = df2.copy()
for col in RP_FEATS:
    lo, hi = df2[col].quantile([WINSOR, 1 - WINSOR])
    n_clipped = ((df2[col] < lo) | (df2[col] > hi)).sum()
    df2_clip[col] = df2[col].clip(lo, hi)
    print(f"{col}: clipped [{lo:.2f}, {hi:.2f}]  ({n_clipped} sites clipped)")

scaler2 = RobustScaler()
X2 = scaler2.fit_transform(df2_clip[RP_FEATS])
for i, col in enumerate(RP_SCALED):
    df2[col] = X2[:, i]

## Build Ward linkage matrix and dendrogram — AEP

In [ ]:
Z2 = linkage(X2, method=LINKAGE_METHOD)

fig, ax = plt.subplots(figsize=(13, 5))
dendrogram(
    Z2, ax=ax,
    truncate_mode='lastp', p=50,
    leaf_rotation=90, leaf_font_size=8,
    color_threshold=0,
    above_threshold_color=rplot.OKABE_ITO[1]
)
ax.set_title(f'AEP Ward dendrogram (last 50 merges)  —  {len(df2):,} sites')
ax.set_xlabel('Site (or cluster size)')
ax.set_ylabel('Ward linkage distance')
plt.tight_layout()
plt.show()

## Silhouette sweep — AEP thresholds

In [ ]:
sil_scores2 = []

for k in sil_k_range:
    labels = fcluster(Z2, k, criterion='maxclust') - 1
    sil_scores2.append(silhouette_score(X2, labels))
    print(f"k={k:2d}  sil={sil_scores2[-1]:.4f}")

best_k2 = list(sil_k_range)[np.argmax(sil_scores2)]
print(f"\nBest k by silhouette: {best_k2}  (score={max(sil_scores2):.4f})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(sil_k_range), sil_scores2, marker='o', color=rplot.OKABE_ITO[1])
ax.axvline(best_k2, color='red', linestyle='--', label=f'best k={best_k2}')
ax.axvline(N_CLUSTERS_AEP, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_AEP}')
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Mean silhouette score')
ax.set_title('Silhouette — AEP Ward')
ax.legend()
plt.tight_layout()
plt.show()

## Elbow plot — AEP thresholds

Total within-cluster variance vs k, derived from the pre-computed Ward linkage matrix.

In [ ]:
n2 = len(X2)
inertia2 = [Z2[:n2 - k, 2].sum() for k in sil_k_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(sil_k_range), inertia2, marker='o', color=rplot.OKABE_ITO[1])
ax.axvline(best_k2, color='red',  linestyle='--', label=f'best sil k={best_k2}')
ax.axvline(N_CLUSTERS_AEP, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_AEP}')
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Total within-cluster variance (inertia)')
ax.set_title('Elbow — AEP Ward')
ax.legend()
plt.tight_layout()
plt.show()

## Hierarchical clustering — AEP thresholds

In [ ]:
hc2 = AgglomerativeClustering(n_clusters=N_CLUSTERS_AEP, linkage=LINKAGE_METHOD)
df2["cluster"] = hc2.fit_predict(X2) + 1

sizes2 = df2["cluster"].value_counts().sort_index()
print(f"N_CLUSTERS={N_CLUSTERS_AEP}  |  linkage={LINKAGE_METHOD}")
print(f"Silhouette: {silhouette_score(X2, df2['cluster']):+.4f}")
print(sizes2.to_string())
print(f"\nMin: {sizes2.min()}  |  Max: {sizes2.max()}  |  Ratio: {sizes2.max()/sizes2.min():.1f}x")

## Cluster map — AEP thresholds

In [ ]:
cluster_colors2 = rplot.cluster_cmap(N_CLUSTERS_AEP).colors

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f'AEP threshold Ward hierarchical clusters  (N={N_CLUSTERS_AEP})\n'
    f'Features: log10 return period — action / flood / moderate / major'
))

for cl in sorted(df2["cluster"].unique()):
    idx = df2["cluster"] == cl
    ax.scatter(
        df2.loc[idx, "longitude"], df2.loc[idx, "latitude"],
        s=20, color=cluster_colors2[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(),
        label=f'C{cl} (n={idx.sum():,})'
    )

ax.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)
plt.tight_layout()
plt.show()

## Cluster profiles — AEP thresholds

In [ ]:
rp_medians2 = df2.groupby("cluster")[RP_FEATS].median()
rp_years2   = 10 ** rp_medians2
rp_years2.columns = [c.replace("_rp", "_yr") for c in rp_years2.columns]
print("Median return period (years) per cluster:")
print(rp_years2.round(1).to_string())

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
labels_aep = ["Action", "Flood", "Moderate", "Major"]
for i, (rp, label) in enumerate(zip(RP_FEATS, labels_aep)):
    for cl in sorted(df2["cluster"].unique()):
        vals = df2.loc[df2["cluster"] == cl, rp]
        axes[i].boxplot(
            vals, positions=[cl], widths=0.6,
            patch_artist=True,
            boxprops=dict(facecolor=cluster_colors2[cl - 1], alpha=0.8),
            medianprops=dict(color='black'),
            flierprops=dict(marker='.', markersize=2, alpha=0.3),
            whiskerprops=dict(linewidth=0.8),
            capprops=dict(linewidth=0.8)
        )
    axes[i].set_title(label)
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel('log10(return period)')
rplot.panel_labels(list(axes))
plt.suptitle('AEP Ward: log10 return period by cluster', y=1.01)
plt.tight_layout()
plt.show()

---
# Comparison 1: Ward vs SKATER — LP3

Load saved SKATER LP3 labels and compare against the Ward solution above.

In [ ]:
# skater_lp3 = read_skater(SKATER_DIR / f"site_regions_skater_lp3_{N_CLUSTERS_LP3}.csv")
# skater_lp3 = skater_lp3.rename(columns={"cluster": "cl_skater"})

# hc_lp3 = df1[["site_no", "cluster"]].rename(columns={"cluster": "cl_hc"})

# cmp1 = hc_lp3.merge(skater_lp3, on="site_no")
# cmp1 = cmp1.merge(df1[["site_no", "latitude", "longitude"] + LP3_FEATS + LP3_SCALED], on="site_no")

# print(f"Sites in Ward    : {len(hc_lp3):,}")
# print(f"Sites in SKATER  : {len(skater_lp3):,}")
# print(f"Sites in both    : {len(cmp1):,}")

In [ ]:
# X1_cmp = cmp1[LP3_SCALED].values

# sil_hc     = silhouette_score(X1_cmp, cmp1["cl_hc"])
# sil_skater = silhouette_score(X1_cmp, cmp1["cl_skater"])
# ari1       = adjusted_rand_score(cmp1["cl_skater"], cmp1["cl_hc"])
# nmi1       = normalized_mutual_info_score(cmp1["cl_skater"], cmp1["cl_hc"])

# print(f"Silhouette — Ward   : {sil_hc:.4f}")
# print(f"Silhouette — SKATER : {sil_skater:.4f}  (spatial cost: {sil_hc - sil_skater:+.4f})")
# print(f"ARI                 : {ari1:.3f}   (0=random, 1=perfect)")
# print(f"NMI                 : {nmi1:.3f}   (0=independent, 1=identical)")

## Side-by-side map — LP3

In [ ]:
# n_cl_hc1 = cmp1["cl_hc"].nunique()
# n_cl_sk1 = cmp1["cl_skater"].nunique()
# cc_hc1   = rplot.cluster_cmap(n_cl_hc1).colors
# cc_sk1   = rplot.cluster_cmap(n_cl_sk1).colors

# fig = plt.figure(figsize=(13, 6))

# ax1 = make_conus_ax(fig, pos=(1, 2, 1), title=f'LP3 Ward  (N={N_CLUSTERS_LP3})')
# for cl in sorted(cmp1["cl_hc"].unique()):
#     idx = cmp1["cl_hc"] == cl
#     ax1.scatter(
#         cmp1.loc[idx, "longitude"], cmp1.loc[idx, "latitude"],
#         s=6, color=cc_hc1[cl - 1], zorder=4,
#         transform=ccrs.PlateCarree(), label=f'C{cl}'
#     )
# ax1.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

# ax2 = make_conus_ax(fig, pos=(1, 2, 2), title=f'LP3 SKATER  (N={N_CLUSTERS_LP3})')
# for cl in sorted(cmp1["cl_skater"].unique()):
#     idx = cmp1["cl_skater"] == cl
#     ax2.scatter(
#         cmp1.loc[idx, "longitude"], cmp1.loc[idx, "latitude"],
#         s=6, color=cc_sk1[cl - 1], zorder=4,
#         transform=ccrs.PlateCarree(), label=f'C{cl}'
#     )
# ax2.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

# rplot.panel_labels([ax1, ax2])
# plt.tight_layout()
# plt.show()

## Spatial agreement map — LP3

Optimal label alignment via the Hungarian algorithm, then sites coloured by whether Ward and SKATER agree.

In [ ]:
# ct1 = pd.crosstab(cmp1["cl_skater"], cmp1["cl_hc"],
#                   rownames=["SKATER"], colnames=["Ward"])
# row_idx1, col_idx1 = linear_sum_assignment(-ct1.values)
# sk_to_hc1 = {ct1.index[r]: ct1.columns[c] for r, c in zip(row_idx1, col_idx1)}

# cmp1["cl_skater_mapped"] = cmp1["cl_skater"].map(sk_to_hc1)
# cmp1["agree"] = cmp1["cl_skater_mapped"] == cmp1["cl_hc"]

# n_agree1 = cmp1["agree"].sum()
# print(f"Overall agreement: {n_agree1:,}/{len(cmp1):,}  ({100*n_agree1/len(cmp1):.1f}%)")
# print(f"\nOptimal SKATER → Ward mapping:")
# for sk, hc in sorted(sk_to_hc1.items()):
#     n_ag = ct1.loc[sk, hc] if hc in ct1.columns else 0
#     n_to = ct1.loc[sk].sum()
#     print(f"  SKATER C{sk} → Ward C{hc}  ({n_ag:,}/{n_to:,} agree, {100*n_ag/n_to:.0f}%)")

# agree1_df    = cmp1[cmp1["agree"]]
# disagree1_df = cmp1[~cmp1["agree"]]

# fig = plt.figure(figsize=(16, 9))
# ax  = make_conus_ax(fig, title=(
#     f"LP3: Ward vs SKATER agreement (optimal label mapping)\n"
#     f"ARI={ari1:.3f}  |  agree={n_agree1:,}/{len(cmp1):,} ({100*n_agree1/len(cmp1):.0f}%)  "
#     f"|  sil cost={sil_hc - sil_skater:+.4f}"
# ))
# ax.scatter(
#     agree1_df.longitude, agree1_df.latitude,
#     s=15, color=rplot.OKABE_ITO[1], alpha=0.6,
#     transform=ccrs.PlateCarree(), zorder=4,
#     label=f"Agree ({len(agree1_df):,})"
# )
# ax.scatter(
#     disagree1_df.longitude, disagree1_df.latitude,
#     s=10, color=rplot.OKABE_ITO[5], alpha=0.9,
#     transform=ccrs.PlateCarree(), zorder=5,
#     label=f"Disagree ({len(disagree1_df):,})"
# )
# ax.legend(markerscale=2, fontsize=9, loc="lower left", framealpha=0.8)
# plt.tight_layout()
# plt.show()

---
# Comparison 2: Ward vs SKATER — AEP thresholds

In [ ]:
# skater_aep = read_skater(SKATER_DIR / f"site_regions_skater_aep_{N_CLUSTERS_AEP}.csv")
# skater_aep = skater_aep.rename(columns={"cluster": "cl_skater"})

# hc_aep = df2[["site_no", "cluster"]].rename(columns={"cluster": "cl_hc"})

# cmp2 = hc_aep.merge(skater_aep, on="site_no")
# cmp2 = cmp2.merge(df2[["site_no", "latitude", "longitude"] + RP_FEATS + RP_SCALED], on="site_no")

# print(f"Sites in Ward    : {len(hc_aep):,}")
# print(f"Sites in SKATER  : {len(skater_aep):,}")
# print(f"Sites in both    : {len(cmp2):,}")

In [ ]:
# X2_cmp = cmp2[RP_SCALED].values

# sil_hc2     = silhouette_score(X2_cmp, cmp2["cl_hc"])
# sil_skater2 = silhouette_score(X2_cmp, cmp2["cl_skater"])
# ari2        = adjusted_rand_score(cmp2["cl_skater"], cmp2["cl_hc"])
# nmi2        = normalized_mutual_info_score(cmp2["cl_skater"], cmp2["cl_hc"])

# print(f"Silhouette — Ward   : {sil_hc2:.4f}")
# print(f"Silhouette — SKATER : {sil_skater2:.4f}  (spatial cost: {sil_hc2 - sil_skater2:+.4f})")
# print(f"ARI                 : {ari2:.3f}")
# print(f"NMI                 : {nmi2:.3f}")

## Side-by-side map — AEP

In [ ]:
# n_cl_hc2 = cmp2["cl_hc"].nunique()
# n_cl_sk2 = cmp2["cl_skater"].nunique()
# cc_hc2   = rplot.cluster_cmap(n_cl_hc2).colors
# cc_sk2   = rplot.cluster_cmap(n_cl_sk2).colors

# fig = plt.figure(figsize=(13, 6))

# ax1 = make_conus_ax(fig, pos=(1, 2, 1), title=f'AEP Ward  (N={N_CLUSTERS_AEP})')
# for cl in sorted(cmp2["cl_hc"].unique()):
#     idx = cmp2["cl_hc"] == cl
#     ax1.scatter(
#         cmp2.loc[idx, "longitude"], cmp2.loc[idx, "latitude"],
#         s=6, color=cc_hc2[cl - 1], zorder=4,
#         transform=ccrs.PlateCarree(), label=f'C{cl}'
#     )
# ax1.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

# ax2 = make_conus_ax(fig, pos=(1, 2, 2), title=f'AEP SKATER  (N={N_CLUSTERS_AEP})')
# for cl in sorted(cmp2["cl_skater"].unique()):
#     idx = cmp2["cl_skater"] == cl
#     ax2.scatter(
#         cmp2.loc[idx, "longitude"], cmp2.loc[idx, "latitude"],
#         s=6, color=cc_sk2[cl - 1], zorder=4,
#         transform=ccrs.PlateCarree(), label=f'C{cl}'
#     )
# ax2.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

# rplot.panel_labels([ax1, ax2])
# plt.tight_layout()
# plt.show()

## Spatial agreement map — AEP

In [ ]:
# ct2 = pd.crosstab(cmp2["cl_skater"], cmp2["cl_hc"],
#                   rownames=["SKATER"], colnames=["Ward"])
# row_idx2, col_idx2 = linear_sum_assignment(-ct2.values)
# sk_to_hc2 = {ct2.index[r]: ct2.columns[c] for r, c in zip(row_idx2, col_idx2)}

# cmp2["cl_skater_mapped"] = cmp2["cl_skater"].map(sk_to_hc2)
# cmp2["agree"] = cmp2["cl_skater_mapped"] == cmp2["cl_hc"]

# n_agree2 = cmp2["agree"].sum()
# print(f"Overall agreement: {n_agree2:,}/{len(cmp2):,}  ({100*n_agree2/len(cmp2):.1f}%)")

# agree2_df    = cmp2[cmp2["agree"]]
# disagree2_df = cmp2[~cmp2["agree"]]

# fig = plt.figure(figsize=(16, 9))
# ax  = make_conus_ax(fig, title=(
#     f"AEP: Ward vs SKATER agreement (optimal label mapping)\n"
#     f"ARI={ari2:.3f}  |  agree={n_agree2:,}/{len(cmp2):,} ({100*n_agree2/len(cmp2):.0f}%)  "
#     f"|  sil cost={sil_hc2 - sil_skater2:+.4f}"
# ))
# ax.scatter(
#     agree2_df.longitude, agree2_df.latitude,
#     s=15, color=rplot.OKABE_ITO[1], alpha=0.6,
#     transform=ccrs.PlateCarree(), zorder=4,
#     label=f"Agree ({len(agree2_df):,})"
# )
# ax.scatter(
#     disagree2_df.longitude, disagree2_df.latitude,
#     s=10, color=rplot.OKABE_ITO[5], alpha=0.9,
#     transform=ccrs.PlateCarree(), zorder=5,
#     label=f"Disagree ({len(disagree2_df):,})"
# )
# ax.legend(markerscale=2, fontsize=9, loc="lower left", framealpha=0.8)
# plt.tight_layout()
# plt.show()

---
## Summary table

In [ ]:
# summary = pd.DataFrame([
#     {
#         "Analysis"     : "LP3",
#         "Method"       : "Ward",
#         "k"            : N_CLUSTERS_LP3,
#         "Silhouette"   : round(sil_hc, 4),
#         "ARI vs SKATER": round(ari1, 3),
#         "NMI vs SKATER": round(nmi1, 3),
#         "Agreement %"  : round(100 * n_agree1 / len(cmp1), 1),
#     },
#     {
#         "Analysis"     : "LP3",
#         "Method"       : "SKATER",
#         "k"            : N_CLUSTERS_LP3,
#         "Silhouette"   : round(sil_skater, 4),
#         "ARI vs SKATER": 1.0,
#         "NMI vs SKATER": 1.0,
#         "Agreement %"  : 100.0,
#     },
#     {
#         "Analysis"     : "AEP",
#         "Method"       : "Ward",
#         "k"            : N_CLUSTERS_AEP,
#         "Silhouette"   : round(sil_hc2, 4),
#         "ARI vs SKATER": round(ari2, 3),
#         "NMI vs SKATER": round(nmi2, 3),
#         "Agreement %"  : round(100 * n_agree2 / len(cmp2), 1),
#     },
#     {
#         "Analysis"     : "AEP",
#         "Method"       : "SKATER",
#         "k"            : N_CLUSTERS_AEP,
#         "Silhouette"   : round(sil_skater2, 4),
#         "ARI vs SKATER": 1.0,
#         "NMI vs SKATER": 1.0,
#         "Agreement %"  : 100.0,
#     },
# ])

# print(summary.to_string(index=False))